# Phase 5 - LightGBM point model

Thin notebook that orchestrates `seercast.training.train_lightgbm` to:

1. Load the supervised feature table (`train_features_ca1.parquet`).
2. For each backtest origin in `BACKTEST.origins`, slice train / valid / test under the strict ML leakage rule (train uses `target_date <= valid_start`, valid is the most-recent 56 days, test is `origin_date == origin`).
3. Fit a Poisson LightGBM with early stopping on the validation slice.
4. Persist the trained models, long predictions, per-origin x horizon scores, and a head-to-head comparison vs. the Phase 3 baselines.
5. Render error-by-horizon, error-by-category, and feature importance.

All real logic lives in `seercast.models.lightgbm_model` and `seercast.training.train_lightgbm`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from seercast.training.train_lightgbm import run as run_lightgbm

REPO_ROOT

## 1. Run the backtest

In [ ]:
result = run_lightgbm()
predictions = result['predictions']
scores = result['scores']
comparison = result['comparison']
models = result['models']
comparison

## 2. Error by horizon

If LightGBM is doing real work, its WAPE should be lower than every baseline at every horizon. Errors typically grow with horizon because farther-out targets are harder.

In [ ]:
from seercast.evaluation.metrics import score_by_group
from seercast.config import ARTIFACTS

# Combine LGBM + baselines, score by (model, horizon).
baseline_preds = pd.read_parquet(ARTIFACTS.baseline_predictions_ca1)
baseline_preds = baseline_preds[baseline_preds['origin_date'].isin(predictions['origin_date'].unique())]
combined = pd.concat([baseline_preds, predictions], ignore_index=True)
by_h = score_by_group(combined, by=['model', 'horizon'])

fig, ax = plt.subplots(figsize=(11, 4))
for name, sub in by_h.groupby('model'):
    ax.plot(sub['horizon'], sub['WAPE'], marker='o', label=name)
ax.set_xlabel('horizon (days ahead)')
ax.set_ylabel('WAPE')
ax.set_title('WAPE by horizon at backtest origins (CA_1)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Error by category

Joining on the base table to attach `cat_id` so we can score by category.

In [ ]:
base = pd.read_parquet(ARTIFACTS.base_table_ca1, columns=['id', 'cat_id']).drop_duplicates('id')
preds_cat = predictions.merge(base, on='id', how='left')
score_by_group(preds_cat, by=['cat_id'])

## 4. Feature importance (last trained model)

Expected: `sales_lag_7`, `sales_lag_28`, `rolling_mean_28`, `rolling_mean_7`, `horizon`, `target_dayofweek`, `target_sell_price`, `zero_sales_rate_28`, `item_id`, `dept_id`, `cat_id` near the top.

In [ ]:
last_model = list(models.values())[-1]
imp = last_model.feature_importance(kind='gain')
imp.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
imp.head(25).iloc[::-1].plot.barh(x='feature', y='gain', ax=ax, legend=False)
ax.set_title('LightGBM feature importance (top 25 by gain)')
plt.tight_layout()
plt.show()

**Next:** Phase 6 - quantile LightGBM (p10/p50/p90) for probabilistic forecasting.